# 실습 1: 신경망 입력을 위한 텐서 연습

## 오늘 할 일 — 75분

이론 1장의 **데이터 표**를 PyTorch 텐서로 다룬다.
수업이 끝나면 표에서 입력과 정답을 고르고, 기본 연산과 행렬 곱을 사용할 수 있다.
신경망을 만드는 코드는 다음 실습에서 `nn.Linear`로 시작한다.

- **대응 이론**: [Ch01 데이터와 모형](../chapters/ch01.qmd)
- **준비 지식**: Python 변수, 리스트, 인덱스. 새로운 텐서 문법은 예제를 보며 익힌다.
- **실행 환경**: Google Colab의 기본 CPU 런타임이면 충분하다.
- 실습은 제출하거나 채점하지 않는다. **직접 해보기**에서 2~4분씩 멈추어 스스로 작성한다.
- `None`을 채우고 확인 셀을 실행한다. `assert`는 결과를 확인하는 코드이므로 수정하지 않는다.
- 막히면 힌트를 먼저 본다. 정답은 문서 끝의 **해설**에 있다.

| 시간 | 내용 |
|---|---|
| 0–8분 | 셀 실행, 텐서 생성, 모양과 자료형 |
| 8–23분 | 행·열 선택과 조건 마스크 |
| 23–38분 | 사칙연산과 오차 계산 |
| 38–50분 | 합·평균과 `dim` |
| 50–65분 | 행렬 곱 `@` |
| 65–75분 | 종합 연습과 정리 |

## 1. 숫자를 텐서에 담기 — 8분

Colab에서 이 노트북을 본인 드라이브에 복사한다. 코드 셀은 **Shift+Enter**로 실행한다.
위에서 만든 변수는 아래 셀에서도 쓴다. 런타임을 다시 시작했다면 위에서부터 다시 실행한다.

In [ ]:
import torch

v = torch.tensor([1, 2, 3], dtype=torch.float32)
print(v)
print(v.shape, v.dtype)

`torch.tensor(...)`는 리스트의 숫자를 텐서로 만든다. `shape`는 축별 크기, `dtype`은 자료형이다.
이번 실습의 실수 입력은 **`dtype=torch.float32`를 명시**해서 만든다.

In [ ]:
A = torch.tensor([[1, 2, 3],
                  [4, 5, 6]], dtype=torch.float32)
print(A)
print(A.shape)          # (2, 3): 행 2개, 열 3개

integer_ids = torch.tensor([0, 1, 2])
print(integer_ids.dtype)
print(integer_ids.float().dtype)    # 이미 있는 텐서의 자료형 바꾸기

실수 값을 계산하는 입력과 정수 번호는 용도가 다르다. 뒤의 분류·텍스트 실습에서는 정수 텐서도 사용한다.

### 직접 해보기 ① — 텐서 만들기

아래 숫자로 **3행 2열 `float32` 텐서** `practice_a`를 만든다.

```text
1  2
3  4
5  6
```

In [ ]:
# ✏️ 직접 채워 보세요
practice_a = None

assert isinstance(practice_a, torch.Tensor), '리스트를 torch.tensor로 바꾸세요.'
assert practice_a.shape == (3, 2), '행과 열의 수를 확인하세요.'
assert practice_a.dtype == torch.float32, 'dtype을 확인하세요.'
assert practice_a.tolist() == [[1., 2.], [3., 4.], [5., 6.]]
print('통과')

힌트: 바깥 리스트 안에 **행별 리스트 세 개**를 넣는다.

## 2. 필요한 행과 열 고르기 — 15분

이론의 사원 8명 자료다. 연봉의 단위를 **천 달러**로 바꾸어 숫자를 읽기 쉽게 했다.
행은 사원 한 명이고, 열은 경력과 연봉이다.

In [ ]:
#                경력(년)  연봉(천 달러)
staff = torch.tensor([[1.5,  31],
                      [2.5,  39],
                      [4.2,  43],
                      [5.1,  49],
                      [6.7,  54],
                      [8.3,  67],
                      [9.5,  92],
                      [13.0, 129]], dtype=torch.float32)
print(staff.shape)

### 인덱싱과 슬라이싱

인덱스는 **0부터** 센다. `a:b`는 a번부터 b번 직전까지이며, `:`는 전부라는 뜻이다.

In [ ]:
print(staff[0])         # 첫 사원의 두 값
print(staff[0, 1])      # 첫 사원의 연봉
print(staff[:3])        # 앞 세 사원
print(staff[:, 0])      # 모든 사원의 경력

경력이 입력 $X$, 연봉이 정답 $y$다. 정답 열을 입력에 넣지 않도록 분리한다.

In [ ]:
experience = staff[:, 0:1]
salary = staff[:, 1:2]
print(experience.shape, salary.shape)

입력과 정답은 **행이 데이터, 열이 변수**인 표 형태로 둔다.
`staff[:, 0:1]`은 경력 열 하나를 표로 고르므로 `(8, 1)`이다.
`staff[:1]`은 첫 사원 한 명을 표로 고르므로 `(1, 2)`다.
신경망에 한 건을 넣을 때도 이처럼 행을 남긴다.

### 조건 마스크

비교 연산은 위치마다 `True` 또는 `False`를 만든다. 이 **마스크**로 원하는 사원만 고른다.
같은 마스크를 입력과 정답에 적용하면 사원 순서가 유지된다.

In [ ]:
mask = staff[:, 0] >= 8
print(mask)
print(staff[mask])
print(salary[mask])

### 직접 해보기 ② — 경력 5년 미만인 사원

`junior_mask`를 만들고, 해당 사원의 **전체 행**과 **연봉**을 각각 고른다.

In [ ]:
# ✏️ 직접 채워 보세요
junior_mask = None
junior_staff = None
junior_salary = None

assert junior_mask is not None and junior_mask.dtype == torch.bool
assert junior_mask.tolist() == [True, True, True, False, False, False, False, False]
assert junior_staff is not None and junior_staff.shape == (3, 2)
assert torch.equal(junior_staff, staff[:3])
assert junior_salary is not None and junior_salary.tolist() == [[31.], [39.], [43.]]
print('통과')

힌트: 조건은 경력으로 만들고, 그 조건으로 `staff`와 `salary`를 각각 고른다.

## 3. 텐서로 계산하기 — 15분

### 숫자 하나에 하던 계산을 여러 값에 적용한다

In [ ]:
print(experience + 1)     # 각 사원의 1년 뒤 경력
print(salary * 1000)      # 천 달러 → 달러
print(salary / 12)        # 연봉 → 월급 (천 달러)

텐서에 숫자 하나를 더하거나 곱하면 각 원소에 같은 계산이 적용된다.

### 같은 위치끼리 계산하기

어떤 모형이 첫 세 사원의 연봉을 아래와 같이 예측했다고 하자.
모형 자체는 다음 실습에서 만든다. 지금은 주어진 예측과 정답의 차이를 계산한다.

In [ ]:
small_y = salary[:3]                         # 정답: (3, 1)
small_pred = torch.tensor([[30.], [40.], [45.]])
error = small_y - small_pred
print(error)                                # [1, -1, -2]를 세로로
print(error ** 2)                            # 각 오차를 제곱

예측과 정답은 모두 `(3, 1)`이다. 각 행은 같은 사원에 해당한다.

### 직접 해보기 ③ — 단위 변환과 오차 계산

1. 첫 세 사원의 연봉을 달러 단위로 바꾸어 `salary_dollars`에 담는다.
2. 새 예측값 `[[32.], [38.], [43.]]`을 `new_prediction`으로 만든다. 단위는 천 달러다.
3. `small_y - new_prediction`을 `new_error`에 담고 제곱한 값을 `squared_error`에 담는다.

In [ ]:
# ✏️ 직접 채워 보세요
salary_dollars = None
new_prediction = None
new_error = None
squared_error = None

assert salary_dollars is not None and salary_dollars.shape == (3, 1)
assert torch.allclose(salary_dollars, torch.tensor([[31000.], [39000.], [43000.]]))
assert new_prediction is not None and new_prediction.shape == small_y.shape
assert torch.allclose(new_prediction, torch.tensor([[32.], [38.], [43.]]))
assert new_error is not None and torch.allclose(new_error, torch.tensor([[-1.], [1.], [0.]]))
assert squared_error is not None and torch.allclose(squared_error, torch.tensor([[1.], [1.], [0.]]))
print('통과')

힌트: `*`는 곱하기, `-`는 빼기, `** 2`는 제곱이다.
먼저 끝났다면 첫 사원 한 명만 `salary[:1]`로 골라 같은 단위 변환을 해 본다.

## 4. 어느 축을 줄일까 — 12분

축 번호는 왼쪽부터 0, 1, …이다. `dim`에는 **계산해서 줄일 축**을 지정한다.
지금처럼 `(행, 열)`인 표에서는 `dim=0`이 행을 모아 열별 결과를, `dim=1`이 열을 모아 행별 결과를 만든다.

In [ ]:
M = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
print(M.sum())          # 전체를 더한다: 스칼라
print(M.sum(dim=0))     # 행 축을 줄인다: (3,)
print(M.sum(dim=1))     # 열 축을 줄인다: (2,)
print(M.mean(dim=0))    # 열별 평균: (3,)

사원 표에서는 열마다 단위가 다르므로 경력과 연봉을 섞어 평균내지 않는다.
열별 평균과 특정 사원들의 평균을 구해 본다.

In [ ]:
print(staff.mean(dim=0))             # 평균 경력, 평균 연봉
print(salary[staff[:, 0] >= 8].mean())  # 경력 8년 이상 사원의 평균 연봉

이론 1장의 **잔차 제곱합**도 제곱과 합으로 계산한다.

In [ ]:
sse = (error ** 2).sum()
print(sse.item())          # 6.0: 원소 하나인 텐서를 Python 숫자로 꺼낸다

### 직접 해보기 ④ — 행별 합과 열별 평균

`M`의 행별 합 `row_sum`, 열별 평균 `col_mean`, 전체 합 `total`을 구한다.

In [ ]:
# ✏️ 직접 채워 보세요
row_sum = None
col_mean = None
total = None

assert row_sum is not None and row_sum.shape == (2,)
assert torch.allclose(row_sum, torch.tensor([6., 15.]))
assert col_mean is not None and col_mean.shape == (3,)
assert torch.allclose(col_mean, torch.tensor([2.5, 3.5, 4.5]))
assert total is not None and total.ndim == 0
assert total.item() == 21.
print('통과')

힌트: 원하는 결과의 모양을 먼저 적고, 사라져야 하는 축을 찾는다.

## 5. 행렬 곱 `@` — 15분

### 원소별 곱과 행렬 곱

신경망 내부에서는 행렬 곱을 사용한다. 먼저 작은 행렬로 `*`와 `@`를 구분한다.

In [ ]:
A = torch.tensor([[1., 2.],
                  [3., 4.]])
B = torch.tensor([[2., 0.],
                  [1., 3.]])
print(A * B)         # 같은 위치끼리 곱하기
print(A @ B)         # 왼쪽의 행과 오른쪽의 열을 곱해서 더하기

`A @ B`의 첫 행, 첫 열 값은 $1\times2+2\times1=4$다.
첫 행, 둘째 열 값은 $1\times0+2\times3=6$이다.
나머지 두 값도 먼저 예상한 뒤 출력과 맞춰 본다.

### 행이 늘어나도 계산 순서는 같다

In [ ]:
A = torch.tensor([[1., 2.],
                  [3., 4.],
                  [5., 6.]])
result = A @ B
print(result)
print(A.shape, B.shape, result.shape)
print(A[:1] @ B)     # 첫 행만 넣어도 같은 순서로 계산한다

$$(n, p)\ @\ (p, q) \longrightarrow (n, q)$$

**왼쪽 행렬의 열 수와 오른쪽 행렬의 행 수가 같아야 한다.**
결과는 왼쪽의 행 수와 오른쪽의 열 수를 갖는다.

### 직접 해보기 ⑤ — 출력 열 하나 만들기

`B_one`을 `[[2.], [3.]]`으로 만들고 `A @ B_one`을 계산한다.
실행하기 전에 결과가 몇 행 몇 열인지 예상한다.

In [ ]:
# ✏️ 직접 채워 보세요
B_one = None
one_column = None
first_row = None

assert B_one is not None and B_one.shape == (2, 1)
assert torch.equal(B_one, torch.tensor([[2.], [3.]]))
assert one_column is not None and one_column.shape == (3, 1)
assert torch.allclose(one_column, torch.tensor([[8.], [18.], [28.]]))
assert first_row is not None and first_row.shape == (1, 1)
assert torch.allclose(first_row, one_column[:1])
print('통과')

힌트: `one_column`에는 전체 행의 계산을, `first_row`에는 `A[:1] @ B_one`을 담는다.
다음 실습에서는 `nn.Linear`가 내부에서 이 행렬 곱과 편향 더하기를 수행한다.

## 6. 종합 연습 — 10분

카페의 네 날짜 자료다. **입력과 정답 선택 → 통계 → 주어진 예측의 오차 계산**을 작성한다.

In [ ]:
#              방문자(십 명)  광고비(만 원)  매출(만 원)
cafe = torch.tensor([[1., 0.,  6.],
                     [2., 1., 12.],
                     [3., 0., 14.],
                     [4., 2., 24.]])
cafe_pred = torch.tensor([[6.], [13.], [14.], [24.]])  # 모형이 예측한 값

1. 앞 두 열을 입력 `cafe_X`, 마지막 열을 정답 `cafe_y`로 고른다. 모양은 각각 `(4, 2)`, `(4, 1)`이다.
2. 입력 변수별 평균 `feature_mean`을 구한다.
3. 예측값 `cafe_pred`와 정답으로 잔차 제곱합 `cafe_sse`를 계산한다.
4. 광고비가 0인 날짜의 실제 매출 평균 `no_ad_mean`을 구한다.

In [ ]:
# ✏️ 직접 채워 보세요
cafe_X = None
cafe_y = None
feature_mean = None
cafe_sse = None
no_ad_mean = None

assert cafe_X is not None and cafe_X.shape == (4, 2)
assert torch.equal(cafe_X, cafe[:, :2])
assert cafe_y is not None and cafe_y.shape == (4, 1)
assert torch.equal(cafe_y, cafe[:, 2:3])
assert feature_mean is not None and torch.allclose(feature_mean, torch.tensor([2.5, 0.75]))
assert cafe_sse is not None and abs(cafe_sse.item() - 1.) < 1e-5
assert no_ad_mean is not None and abs(no_ad_mean.item() - 10.) < 1e-5
print('통과')

먼저 끝났다면 `cafe_X[:1]`로 첫 날짜의 입력만 고른다. 모양이 `(1, 2)`인지 확인한다.

## 이번 주에 손에 익힐 문법

| 할 일 | 코드 |
|---|---|
| 실수 텐서 생성 | `torch.tensor(..., dtype=torch.float32)` |
| 모양·자료형 확인 | `X.shape`, `X.dtype` |
| 행·열 선택 | `X[:1]`, `X[:, 0:1]`, `X[:, :2]` |
| 조건 선택 | `mask = X[:, 0] > 2`, `X[mask]` |
| 줄일 축 지정 | `X.sum(dim=0)`, `X.mean(dim=1)` |
| 원소별 곱 / 행렬 곱 | `A * B` / `A @ B` |
| 스칼라 꺼내기 | `sse.item()` |

**마무리 질문**: `(5, 3) @ (3, 2)`의 결과 모양은 무엇인가? 첫 행만 계산하려면 왼쪽 입력을 어떻게 고르는가?

[다음 실습: 퍼셉트론을 쌓아 신경망 만들기](lab02.qmd)에서는 준비한 입력을 `nn.Linear`에 넣고 활성화 함수와 여러 층을 연결한다.

---

## 해설 — 먼저 직접 풀고 확인하기

해설의 코드는 참고용이다. 복사할 때는 해당 문제 셀의 `None` 자리에 옮기고, 그 문제의 확인 코드로 검산한다.

### ① 텐서 만들기

```python
practice_a = torch.tensor([[1, 2], [3, 4], [5, 6]], dtype=torch.float32)
```

### ② 조건 선택

```python
junior_mask = staff[:, 0] < 5
junior_staff = staff[junior_mask]
junior_salary = salary[junior_mask]
```

### ③ 단위 변환과 오차

```python
salary_dollars = small_y * 1000
new_prediction = torch.tensor([[32.], [38.], [43.]])
new_error = small_y - new_prediction
squared_error = new_error ** 2
```

### ④ 축별 계산

```python
row_sum = M.sum(dim=1)
col_mean = M.mean(dim=0)
total = M.sum()
```

### ⑤ 행렬 곱

```python
B_one = torch.tensor([[2.], [3.]])
one_column = A @ B_one
first_row = A[:1] @ B_one
```

### 종합 연습

```python
cafe_X = cafe[:, :2]
cafe_y = cafe[:, 2:3]
feature_mean = cafe_X.mean(dim=0)
cafe_sse = ((cafe_y - cafe_pred) ** 2).sum()
no_ad_mean = cafe[cafe[:, 1] == 0, 2].mean()
```